In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [61]:
# Replace with the actual paths to your CSV files

dx_df = pd.read_csv('data/DXSUM_17Feb2026.csv')
dem  = pd.read_csv('data/PTDEMOG_17Feb2026.csv') 
adas = pd.read_csv('data/ADAS_17Feb2026.csv')
medhist = pd.read_csv('data/MEDHIST_17Feb2026.csv')
recmhist = pd.read_csv('data/RECMHIST_17Feb2026.csv')
reccmeds = pd.read_csv('data/RECCMEDS_16Mar2026.csv')
backmeds = pd.read_csv('data/BACKMEDS_16Mar2026.csv')
demographics = pd.read_csv('data/PTDEMOG_17Feb2026.csv')
apoe_df = pd.read_csv("data/APOERES_17Feb2026.csv")
apoe_df["CARRIER"] = apoe_df["GENOTYPE"].isin(["2/4","3/4","4/4"])
apoe_df["HOMO"] = apoe_df["GENOTYPE"].isin(["4/4"])

/var/folders/g_/qzn_bmsd7v9fwp049f83fwtr0000gp/T/ipykernel_18372/4255858013.py:8: DtypeWarning: Columns (8,11,13,14,17,21,22,33,34) have mixed types. Specify dtype option on import or set low_memory=False.
  reccmeds = pd.read_csv('data/RECCMEDS_16Mar2026.csv')


In [62]:
print(recmhist["RID"].nunique())
print(adas["RID"].nunique())
print(demographics["RID"].nunique())
print(demographics.columns)


2484
3027
4946
Index(['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'PTSOURCE',
       'PTGENDER', 'PTDOB', 'PTDOBYY', 'PTHAND', 'PTMARRY', 'PTEDUCAT',
       'PTWORKHS', 'PTWORK', 'PTNOTRT', 'PTRTYR', 'PTHOME', 'PTTLANG',
       'PTPLANG', 'PTADBEG', 'PTCOGBEG', 'PTADDX', 'PTETHCAT', 'PTRACCAT',
       'PTIDENT', 'PTORIENT', 'PTORIENTOT', 'PTENGSPK', 'PTNLANG',
       'PTENGSPKAGE', 'PTCLANG', 'PTLANGSP', 'PTLANGWR', 'PTSPTIM',
       'PTSPOTTIM', 'PTLANGPR1', 'PTLANGSP1', 'PTLANGRD1', 'PTLANGWR1',
       'PTLANGUN1', 'PTLANGPR2', 'PTLANGSP2', 'PTLANGRD2', 'PTLANGWR2',
       'PTLANGUN2', 'PTLANGPR3', 'PTLANGSP3', 'PTLANGRD3', 'PTLANGWR3',
       'PTLANGUN3', 'PTLANGPR4', 'PTLANGSP4', 'PTLANGRD4', 'PTLANGWR4',
       'PTLANGUN4', 'PTLANGPR5', 'PTLANGSP5', 'PTLANGRD5', 'PTLANGWR5',
       'PTLANGUN5', 'PTLANGPR6', 'PTLANGSP6', 'PTLANGRD6', 'PTLANGWR6',
       'PTLANGUN6', 'PTLANGTTL', 'PTETHCATH', 'PTASIAN', 'PTOPI', 'PTBORN',
       'PTBIRPL', 'PTIMMAGE', 'PTIMMWHY', 'PTBI

In [63]:
def visit_to_months(v):
    if pd.isna(v):
        return np.nan
    v = v.lower()
    if v in ["bl", "sc"]:
        return 0
    if v.startswith("m"):
        return float(v[1:])  # strip the 'm'
    if v.startswith("v"):
        return float(v[1:])  # strip the 'v'
    
    return np.nan

baseline_codes = ["bl", "4_bl"]
adas["time_months"] = adas["VISCODE"].apply(visit_to_months)

baseline_dates = (
    adas[adas["VISCODE"].isin(baseline_codes)]
    .groupby("RID")["VISDATE"]
    .min()
)

adas_df = adas.merge(
    baseline_dates.rename("baseline_date"),
    on="RID",
    how="left"
)

adas_df["days_since_bl"] = (
    pd.to_datetime(adas_df["VISDATE"]) -
    pd.to_datetime(adas_df["baseline_date"])
)

In [64]:
# Merge ADAS scores with diagnosis data
adas_AD_dx = pd.merge(adas_df, dx_df[["RID", "VISCODE", "DIAGNOSIS"]], on=["RID", "VISCODE"], how="left")
adas_AD_dx['VISDATE'] = pd.to_datetime(adas_AD_dx['VISDATE'])


### Finding comorbidities with string matching:

In [65]:
UTI_KEYWORDS = [
    "urinary tract infection",
    "uti",
]

GI_KEYWORDS = [
    "diarrhea",
    "constipation",
    "halitosis",
    "fecal incontinence",
    "abnormal bowel movement",
    "abnormal bowel sounds",
    "encopresis",
]

SLEEP_KEYWORDS = [
    "insomnia",
    "sleep disorder",
    "sleep disturbance",
    "poor sleep",
    "sleep apnea",
    "obstructive sleep apnea",
    "osa",
    "hypersomnia",
    "sleep fragmentation",
    "restless legs",
    "bruxism",
    "parasomnia",
    "narcolepsy", 
]

CHRONIC_GI_KEYWORDS = [
    "chronic diarrhea",
    "chronic constipation",
    "frequent diarrhea",
    "frequent constipation",
    "occasional diarrhea",
    "occasional constipation",
    "intermittent diarrhea",
    "intermittent constipation",
    "history of diarrhea",
    "history of constipation",
]


In [66]:
SPECIFIC_KEYWORDS = {
    "UTI": UTI_KEYWORDS,
    "GI": GI_KEYWORDS,
    "Sleep": SLEEP_KEYWORDS,
}

# Function to standardise and classify specific conditions based on keywords
def classify_specific(desc):

    if pd.isna(desc):
        return None
    
    
    d = desc.lower()
    d = d.replace("hx of", "history of")
    d = d.replace("h/o", "history of")
    
    for category, keywords in SPECIFIC_KEYWORDS.items():
        if any(k in d for k in keywords):
            return category
    
    return None

recmhist["specific_flag"] = recmhist["MHDESC"].apply(classify_specific)

In [67]:
# Following the work of Bu et al. (2020), we can classify conditions into CNS vs Peripheral based on keywords in the description. This is a simple heuristic approach and may not be perfect, but it can help us identify common comorbidities.

CNS_KEYWORDS = {
    'cerebrovascular': ['stroke', 'cerebrovascular', 'tia', 'transient ischemic', 'cva'],
    'insomnia': ['insomnia', 'sleep disorder'],
    'anxiety': ['anxiety', 'anxious'],
    'depression': ['depression', 'depressive', 'depressed'],
    'head_injury': ['head injury', 'tbi', 'traumatic brain', 'concussion', 'head trauma']
}

PERIPHERAL_KEYWORDS = {
    'hypertension': ['hypertension', 'high blood pressure', 'htn'],
    'hyperlipidemia': ['hyperlipidemia', 'hypercholesterolemia', 'high cholesterol', 'dyslipidemia'],
    'diabetes': ['diabetes', 'diabetic', 'dm', 'niddm', 'iddm'],
    'atrial_fibrillation': ['atrial fibrillation', 'afib', 'a fib', 'a-fib'],
    'coronary': ['coronary', 'ischemic heart', 'ihd', 'cad', 'coronary artery', 'myocardial infarction', 'mi', 'heart attack'],
    'anemia': ['anemia', 'anaemia'],
    'hypothyroid': ['hypothyroid', 'thyroid'],
    'skin_inflammatory': ['psoriasis', 'eczema', 'dermatitis', 'skin inflammation'],
    'pulmonary': ['copd', 'asthma', 'pulmonary', 'emphysema', 'chronic bronchitis', 'respiratory', 'lung disease'],
    'kidney': ['kidney', 'renal', 'ckd', 'chronic kidney'],
    'hepatitis': ['hepatitis', 'liver disease', 'cirrhosis'],
    'osteoporosis': ['osteoporosis', 'bone density'],
    'hearing_loss': ['hearing loss', 'deaf', 'hearing impair'],
    'cancer': ['cancer', 'malignancy', 'carcinoma', 'tumor', 'neoplasm'],
    'gastrointestinal': ['gastro', 'ulcer', 'reflux', 'gerd', 'ibs', 'crohn', 'colitis', 'diverticulitis'],
    'cataract': ['cataract'],
}

def classify_condition_detailed(desc):
    """
    Classify condition into CNS, Peripheral, or Other
    Returns tuple: (category, specific_condition)
    """
    if pd.isna(desc):
        return None, None
    
    d = desc.lower()
    
    # Check CNS conditions
    for condition, keywords in CNS_KEYWORDS.items():
        if any(k in d for k in keywords):
            return "CNS", condition
    
    # Check Peripheral conditions
    for condition, keywords in PERIPHERAL_KEYWORDS.items():
        if any(k in d for k in keywords):
            return "Peripheral", condition
    
    return "Other", None

recmhist["comorb_category"] = recmhist["MHDESC"].apply(
    lambda x: classify_condition_detailed(x)[0]
)
recmhist["specific_condition"] = recmhist["MHDESC"].apply(
    lambda x: classify_condition_detailed(x)[1]
)

In [68]:
recmhist_bl = recmhist[recmhist["VISCODE"].isin(["v01", "sc"])]

In [69]:
comorb_counts = (
    recmhist_bl[recmhist_bl["comorb_category"].isin(["CNS", "Peripheral"])]
    .groupby(["RID", "VISCODE", "comorb_category", "specific_condition"])
    .size()
    .reset_index(name="count")
    .groupby(["RID", "VISCODE", "comorb_category"])
    .agg(
        n_conditions=("specific_condition", "nunique"),
        total_entries=("count", "sum")
    )
    .reset_index()
)

# Create wide format
comorb_wide = comorb_counts.pivot_table(
    index=["RID", "VISCODE"],
    columns="comorb_category",
    values="n_conditions",
    fill_value=0
).reset_index()

# Calculate total multimorbidity burden
comorb_wide["total_conditions"] = (
    comorb_wide.get("CNS", 0) + 
    comorb_wide.get("Peripheral", 0)
)

# Categorize burden as in the paper
def categorize_burden(n):
    if n <= 2:
        return "Low"
    elif n <= 5:
        return "Medium"
    else:
        return "High"

comorb_wide["burden_category"] = comorb_wide["total_conditions"].apply(categorize_burden)

# For CNS burden: 0 vs 1+ 
comorb_wide["CNS_burden"] = comorb_wide["CNS"].apply(lambda x: "0" if x == 0 else "1+")

# For Peripheral burden: 0-1 vs 2+ 
comorb_wide["Peripheral_burden"] = comorb_wide["Peripheral"].apply(
    lambda x: "0-1" if x <= 1 else "2+"
)

In [70]:
print(medhist.columns)

Index(['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'MHSOURCE',
       'MHPSYCH', 'MH2NEURL', 'MH3HEAD', 'MH4CARD', 'MH5RESP', 'MH6HEPAT',
       'MH7DERM', 'MH8MUSCL', 'MH9ENDO', 'MH10GAST', 'MH11HEMA', 'MH12RENA',
       'MH13ALLE', 'MH14ALCH', 'MH14AALCH', 'MH14BALCH', 'MH14CALCH',
       'MH15DRUG', 'MH15ADRUG', 'MH15BDRUG', 'MH16SMOK', 'MH16ASMOK',
       'MH16BSMOK', 'MH16CSMOK', 'MH17MALI', 'MH18SURG', 'MH19OTHR', 'ID',
       'SITEID', 'USERDATE', 'USERDATE2', 'update_stamp'],
      dtype='object')


### Incorporating broad GI and UTI definitions

In [71]:
# Relevant MEDHIST binary columns and their mappings
# 1 = present, 0 = absent, 2 = unknown (treat as 0)
MEDHIST_FLAG_MAP = {
    "MH10GAST": "GI_broad",          # Gastrointestinal disorders
    "MH12RENA": "UTI_broad",         # Renal/genitourinary disorders
}

# Keep only the columns we need plus identifiers
medhist_cols = ["RID", "VISCODE"] + [k for k in MEDHIST_FLAG_MAP if MEDHIST_FLAG_MAP[k] is not None]
medhist_sub = medhist[medhist_cols].copy()

# Recode: treat 2 (unknown) as 0, keep 1 as 1
for col in [k for k in MEDHIST_FLAG_MAP if MEDHIST_FLAG_MAP[k] is not None]:
    medhist_sub[col] = medhist_sub[col].apply(lambda x: 1 if x == 1 else 0)

medhist_sub = medhist_sub.rename(columns={k: v for k, v in MEDHIST_FLAG_MAP.items() if v is not None})

print(medhist_sub["UTI_broad"].value_counts())
sub_bl = medhist_sub[medhist_sub["VISCODE"].isin(['sc','v01'])]
print(sub_bl["UTI_broad"].value_counts())

UTI_broad
0    1745
1    1338
Name: count, dtype: int64
UTI_broad
0    1263
1     895
Name: count, dtype: int64


In [72]:
specific_flags_visit = (
    recmhist[recmhist["specific_flag"].notna()]
    .assign(flag=1)
    .pivot_table(
        index=["RID", "VISCODE"],
        columns="specific_flag",
        values="flag",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

comorb_wide_new = comorb_wide.merge(
    specific_flags_visit,
    on=["RID", "VISCODE"],
    how="left"
)

# Replace NaNs with 0s for the specific flags
for col in ["UTI", "GI", "Sleep"]:
    if col in comorb_wide_new.columns:
        comorb_wide_new[col] = comorb_wide_new[col].fillna(0)

comorb_wide_to_merge = comorb_wide_new.copy()

In [73]:
print("comorb_wide_new VISCODE samples:", comorb_wide_new["VISCODE"].unique()[:10])
print("medhist VISCODE samples:", medhist["VISCODE"].unique()[:10])
print("comorb_wide_new RID dtype:", comorb_wide_new["RID"].dtype)
print("medhist RID dtype:", medhist["RID"].dtype)

comorb_wide_new VISCODE samples: ['sc' 'v01']
medhist VISCODE samples: ['sc' 'f' 'm48' 'm36' 'm60' 'v01' 'v06']
comorb_wide_new RID dtype: int64
medhist RID dtype: int64


In [74]:
# merge broad definitions into the other comorbidity flags 
comorb_wide_updated = comorb_wide_new.merge(
    sub_bl[["RID", "VISCODE", "GI_broad", "UTI_broad"]],
    on=["RID", "VISCODE"],
    how="left",
)

In [75]:
# Examine the free text descriptions of GI participants from final cohort

GI_RIDs = [1394, 1371, 1351, 1265, 1217, 1171, 1081,  941,  855,  839,  790,
        784,  754,  724,  625,  605,  474,  467,  372,  150, 2248, 2398,
       2403, 4015, 4102, 4209, 4250, 4422, 4379, 4404, 4589, 4675, 4657,
       4730, 4707, 4905, 5119]

comorb_GI = comorb_wide_updated[comorb_wide_updated["RID"].isin(GI_RIDs)]
recmhist_GI = recmhist[recmhist["RID"].isin(GI_RIDs)]
for desc in recmhist_GI["MHDESC"]:
    if 'constipation' in desc.lower() or 'diarrhea' in desc.lower():
       print(desc)



diarrhea related to aricept --/--/2003
Diarrhea and constipation -resolved in 2004
Constipation 2005
Constipation (2003)
Drug-induced Diarrhea, 1999, Immodium
Occational diarrhea 2004, secondary to Aricept
Diarrhea (2005)
Chronic constipation since 1950.
Frequent constipation since April 2006
Diarrhea.  ONSET:  APRIL 2005
Constipation.  Onset:  APRIL 2005
constipation- Onset date 1945
gastric reflux (1998), occasional diarrhea (2005)
Constipation. Onset SEPTEMBER 2006 - Ongoing
Occasional Constipation (Age 12). Onset: 1940
hemorrhoids and constipation cured with prune juice.started in her 20's.
Intermittent Diarrhea (Frequently) Possibly Related to Medication (Aricept). Onset: JANUARY 2006 - Ongoing
Occasional Constipation. Onset: 2001 - Ongoing
polymyalgia Rheumatica - onset 1980, episodic, once every 3-4 years. Symptoms of stomach ache, diarrhea, headaches and fainting due to hypotension.
Occasional Constipation. Onset: 2001
diarrhea onset --/--/1991
diarrhea; onset 09/--/10
Diarrhea

### Cohort construction

In [76]:
demographics_unique = demographics[["RID", "PTDOB", "PTGENDER", "PTEDUCAT",'PTETHCAT', 'PTRACCAT']].drop_duplicates(subset=["RID"])

# Merge ADAS test scores with demographics
AD_dem = adas_AD_dx.merge(
    demographics_unique,
    on="RID",
    how="left",
    validate="many_to_one"
)

print(f"Number of individuals with demographic data and cognitive data: {AD_dem['RID'].nunique()}")
print(comorb_wide_updated.columns)
comorb_visit = comorb_wide_updated[[
    "RID", "total_conditions", "CNS", "Peripheral",
    "UTI", "GI", "Sleep", "UTI_broad", "GI_broad"
]].drop_duplicates(subset=["RID"])

# Merge the baseline medical data with the ADAS and demographics merged dataframe
AD = AD_dem.merge(
    comorb_visit,
    on="RID",
    how="left",
    validate="many_to_one"
)
AD.dropna(subset=["total_conditions"], inplace=True)
print(f"Number of individuals with comorbidity data: {AD['RID'].nunique()}" )

Number of individuals with demographic data and cognitive data: 3027
Index(['RID', 'VISCODE', 'CNS', 'Peripheral', 'total_conditions',
       'burden_category', 'CNS_burden', 'Peripheral_burden', 'GI', 'Sleep',
       'UTI', 'GI_broad', 'UTI_broad'],
      dtype='object')
Number of individuals with comorbidity data: 1670


In [77]:
# Add baseline age and ADAS13 score to all later visits
ad_start = AD[['RID', 'VISCODE', 'VISDATE', 'TOTAL13', 'PTGENDER', 'PTDOB', 'PTEDUCAT']].copy()
first_entries = (
    ad_start
    .sort_values('VISDATE')
    .groupby('RID')
    .first()
    .reset_index()
)
print(first_entries['VISCODE'].value_counts())
first_entries['VISDATE'] = pd.to_datetime(first_entries['VISDATE'])
first_entries['PTDOB'] = pd.to_datetime(first_entries['PTDOB'])

# Calculate age at conversion (in years)
first_entries['age_at_baseline'] = (
    (first_entries['VISDATE'] - first_entries['PTDOB'])
    .dt.days / 365.25
)

ad_start = first_entries.rename(columns={
    'VISDATE': 'Study_start_date',
    'TOTAL13': 'TOTAL13_AD_start'
})

AD_new = AD.merge(ad_start[['RID', 'Study_start_date', 'TOTAL13_AD_start', 'age_at_baseline']], on='RID', how='left')
print(AD_new["RID"].nunique())


VISCODE
bl     912
v03    758
Name: count, dtype: int64
1670


/var/folders/g_/qzn_bmsd7v9fwp049f83fwtr0000gp/T/ipykernel_18372/1103482406.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  first_entries['PTDOB'] = pd.to_datetime(first_entries['PTDOB'])


In [78]:
model_vars = [
    "TOTAL13",
    "TOTAL13_AD_start",
    "VISCODE",
    "VISDATE",
    "PHASE",
    "days_since_entry",
    "age_at_baseline",
    "PTGENDER",
    'PTETHCAT', 
    'PTRACCAT',
    "CARRIER",
    "HOMO",
    "PTEDUCAT",
    "CNS",
    "total_conditions",
    "Peripheral",
    "RID",
    "UTI", "GI", "Sleep", 
    "UTI_broad", "GI_broad"
]

In [79]:
# Merge APOE carrier status and homozygosity into the ADAS and demographics merged dataframe
AD_df = AD_new.merge(apoe_df[["RID", "CARRIER", "HOMO"]], on="RID", how="left")

AD_df['VISDATE'] = pd.to_datetime(AD_df['VISDATE'])
AD_df['Study_start_date'] = pd.to_datetime(AD_df['Study_start_date'])
AD_df['days_since_entry'] = (AD_df['VISDATE'] - AD_df['Study_start_date']).dt.days
print(f'Before dropping EO individuals: {AD_df["RID"].nunique()}')

AD_no_EO = AD_df[AD_df['age_at_baseline'] >= 65]
print(f'After dropping EO individuals: {AD_no_EO["RID"].nunique()}')

AD_model = AD_no_EO[model_vars].dropna()
AD_model["time_years"] = AD_model["days_since_entry"] / 365.25

print(AD_model["RID"].nunique())


Before dropping EO individuals: 1670
After dropping EO individuals: 1479
1469


### Add in CSF data:

In [80]:
CSF = pd.read_csv("data/UPENNBIOMK_ROCHE_ELECSYS_19Feb2026.csv")
print(CSF.columns)

CSF["PTAU_ABETA42"] = CSF["PTAU"] / CSF["ABETA42"]
#CSF["ABETA"] = CSF["ABETA40"] / CSF["ABETA42"] # Can't use due to high sparsity of ABETA40 values

CSF_bl = CSF[CSF["VISCODE2"] == "bl"]
CSF_bl["CSF_date"] = CSF_bl["EXAMDATE"]


Index(['PHASE', 'PTID', 'RID', 'VISCODE2', 'EXAMDATE', 'BATCH', 'RUNDATE',
       'ABETA40', 'ABETA42', 'TAU', 'PTAU', 'COMMENT', 'update_stamp'],
      dtype='object')


/var/folders/g_/qzn_bmsd7v9fwp049f83fwtr0000gp/T/ipykernel_18372/60515919.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CSF_bl["CSF_date"] = CSF_bl["EXAMDATE"]


In [81]:
not_na_AB = CSF_bl["ABETA42"].notna().sum()
print(f"Non-missing values for AB42: {not_na_AB}")

not_na_AB40 = CSF_bl["ABETA40"].notna().sum()
print(f"Non-missing values for AB40: {not_na_AB40}")

Non-missing values for AB42: 1619
Non-missing values for AB40: 468


In [82]:
CSF_AD = AD_model.merge(
    CSF_bl[["RID", "VISCODE2", "PTAU", "ABETA42", "PTAU_ABETA42", "CSF_date"]],
    on=["RID"],
    how="left")
CSF_AD["time_sq"] = CSF_AD["time_years"] ** 2

CSF_AD = CSF_AD.dropna(subset=["TOTAL13"])
CSF_AD = CSF_AD.dropna(subset=["PTAU_ABETA42"])
CSF_AD = CSF_AD.dropna(subset=["PTAU"])
CSF_AD = CSF_AD.dropna(subset=["ABETA42"])

print(CSF_AD["RID"].nunique())

1015


In [83]:
print(CSF_AD.groupby(['RID', 'VISCODE']).size().sort_values(ascending=False).head())

RID   VISCODE
3     bl         1
4399  v11        1
      v03        1
      init       1
4396  v21        1
dtype: int64


In [84]:
# Save the final dataset for modelling in 2-MLM_comparisons.ipynb

CSF_AD.to_csv('data/CSF_AD.csv', index=False)

In [85]:
total = CSF_AD["RID"].nunique()

groups = {
    "UTI":       CSF_AD["UTI"] == 1,
    "UTI_broad": CSF_AD["UTI_broad"] == 1,
    "GI":        CSF_AD["GI"] == 1,
    "GI_broad":  CSF_AD["GI_broad"] == 1,
    "Sleep":     CSF_AD["Sleep"] == 1,
}

for name, mask in groups.items():
    n = CSF_AD[mask]["RID"].nunique()
    print(f"{name}: n={n} ({100*n/total:.1f}% of cohort)")

print(f"\nTotal cohort: n={total}")

UTI: n=28 (2.8% of cohort)
UTI_broad: n=458 (45.1% of cohort)
GI: n=84 (8.3% of cohort)
GI_broad: n=486 (47.9% of cohort)
Sleep: n=183 (18.0% of cohort)

Total cohort: n=1015


### Explore Sleep Medications 

In [86]:
# ── Strategy: layer all three sources ────────────────────────────────────
#
# 1. MEDHIST  → pre-study sleep disorder diagnosis (you already use this
#               for your Sleep flag — confirms the condition was present)
#
# 2. BACKMEDS → structured medication flags at each visit
#               KEYMED column contains pipe-separated codes e.g. "4:05:06"
#               Check data dictionary for which code = which drug class
#               Good for: cholinesterase inhibitors, BP meds — less useful
#               for sleep meds specifically
#               MISSING: ADNI1 patients entirely
#
# 3. RECCMEDS → free-text concurrent medications at each visit
#               CMMED column: drug name (messy, needs regex cleaning)
#               CMREASON column: reason prescribed
#               Best source for sleep medications specifically
#               Covers ADNI1 onwards
#               Use the find_orexin_antagonists.py script on this

# ── Quick check: what KEYMED values exist in BACKMEDS ────────────────────
print("=== BACKMEDS KEYMED unique values ===")
print(backmeds["KEYMED"].value_counts().head(30))
# These will be pipe-separated codes — cross-reference with data dictionary
# to see if any sleep medications are tracked

# ── Check ADNI phase coverage ─────────────────────────────────────────────
print("\n=== BACKMEDS visit codes (phase coverage) ===")
print(backmeds["VISCODE"].value_counts().head(20))

# ── Check RECCMEDS for sleep drug mentions ────────────────────────────────
reccmeds["med_lower"] = reccmeds["CMMED"].fillna("").str.lower()

sleep_pattern = (
    "zolpidem|ambien|zopiclone|eszopiclone|zaleplon|"
    "temazepam|triazolam|nitrazepam|lorazepam|"
    "melatonin|ramelteon|trazodone|mirtazapine|"
    "quetiapine|doxepin|diphenhydramine|hydroxyzine|"
    "suvorexant|belsomra|lemborexant|dayvigo"
)

sleep_pattern = ("trazodone")

sleep_reccmeds = reccmeds[reccmeds["med_lower"].str.contains(sleep_pattern, na=False)]

print("\n=== Sleep medications in RECCMEDS ===")
print(f"Rows: {len(sleep_reccmeds)}, Patients: {sleep_reccmeds['RID'].nunique()}")
print(sleep_reccmeds["CMMED"].value_counts().head(20))

# ── Per-patient flag for merging ──────────────────────────────────────────
sleep_med_patients = (
    sleep_reccmeds.groupby("RID")
    .agg(
        sleep_med_ever    = ("CMMED", "count"),
        sleep_med_list    = ("CMMED", lambda x: list(x.unique())),
        earliest_sleep_med = ("CMBGN", "min")
    )
    .reset_index()
)
sleep_med_patients["on_sleep_med"] = True


=== BACKMEDS KEYMED unique values ===
KEYMED
0          8376
6          1937
1           966
1|4         491
1|6         437
1|4|6       355
7           312
6|7         291
3           199
4           126
5           119
1|7          70
3|4|6        69
3|6          69
5|6          65
3|4          65
4|6          65
1|6|7        53
1|4|7        52
4|5          52
4|5|6        48
1|4|6|7      45
4|5|6|7      25
4|6|7        18
4|7          16
3|7           8
5|6|7         6
3|6|7         6
6|0           5
1|3|6         5
Name: count, dtype: int64

=== BACKMEDS visit codes (phase coverage) ===
VISCODE
sc        1307
v01       1128
v11        949
4_sc       852
v21        845
bl         826
v12        790
v03        788
v05        719
y2         643
4_bl       591
y1         546
4_init     504
v31        500
v41        473
init       439
v06        385
y4         342
v07        341
4_m12      279
Name: count, dtype: int64

=== Sleep medications in RECCMEDS ===
Rows: 261, Patients: 196
CMME